<div dir="rtl">

# الترم ٣ · الوحدة ٣: الذكاء الاصطناعي في الروبوتات

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-3/10-robot-gridworld.ipynb)

روبوت توصيل في مخزن لازم يوصل للهدف من غير ما يخبط في الحيطان. هنحل المشكلة بطريقتين:

1. **التخطيط (Planning):** الروبوت عارف الخريطة، فبيحسب أقصر طريق بخوارزمية **BFS**.
2. **التعلم بالتجربة (Reinforcement Learning):** الروبوت **مش عارف** الخريطة. بيجرب ويغلط وياخد **مكافأة** أو **عقاب**، لحد ما يتعلم. ده اسمه **Q-Learning**.

</div>

In [ ]:
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

<div dir="rtl">

## ١. الخريطة
`S` = البداية، `G` = الهدف، `#` = حيطة، `.` = طريق فاضي.

</div>

In [ ]:
MAP = [
    "S...#.....",
    ".##.#.###.",
    ".#........",
    ".#.###.##.",
    ".#.....#..",
    ".####.##.#",
    "......#..G",
]
grid = np.array([list(row) for row in MAP])
H, W = grid.shape
start = tuple(np.argwhere(grid == "S")[0])
goal = tuple(np.argwhere(grid == "G")[0])
ACTIONS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}


def is_free(pos):
    r, c = pos
    return 0 <= r < H and 0 <= c < W and grid[r, c] != "#"


def draw(path=None, values=None, title=""):
    walls = (grid == "#").astype(float)
    fig, ax = plt.subplots(figsize=(W * 0.6, H * 0.6))
    if values is not None:
        shown = np.where(walls == 1, np.nan, values)
        ax.imshow(shown, cmap="viridis")
    ax.imshow(np.ma.masked_where(walls == 0, walls), cmap=ListedColormap(["#374151"]))
    if path:
        ys, xs = zip(*path)
        ax.plot(xs, ys, color="#F59E0B", linewidth=3)
    for (r, c), label, color in [(start, "S", "#2563EB"), (goal, "G", "#DC2626")]:
        ax.add_patch(plt.Circle((c, r), 0.4, color=color))
        ax.text(c, r, label, ha="center", va="center", color="white", fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    plt.show()

draw(title="The warehouse")

<div dir="rtl">

## ٢. التخطيط: BFS (Breadth-First Search)

BFS بيستكشف كل الخانات اللي على بعد ١ خطوة، وبعدين ٢، وبعدين ٣... زي موجة بتتمد. أول ما يلمس الهدف، يبقى لقى **أقصر طريق**.

</div>

In [ ]:
def bfs(start, goal):
    queue = deque([start])
    came_from = {start: None}
    while queue:
        current = queue.popleft()
        if current == goal:
            break
        for dr, dc in ACTIONS.values():
            nxt = (current[0] + dr, current[1] + dc)
            if is_free(nxt) and nxt not in came_from:
                came_from[nxt] = current
                queue.append(nxt)
    if goal not in came_from:
        return None
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    return path[::-1]

path = bfs(start, goal)
print("Shortest path length:", len(path) - 1, "steps")
draw(path, title="BFS shortest path")

<div dir="rtl">

## ٣. Reinforcement Learning: الروبوت بيتعلم بالتجربة

الروبوت دلوقتي **مش شايف الخريطة**. القواعد:
- كل خطوة: **-1** (علشان يستعجل).
- خبط في حيطة: **-5** ويفضل مكانه.
- وصل للهدف: **+50**.

الروبوت بيحفظ جدول اسمه **Q-table**: لكل خانة ولكل حركة، "الحركة دي بتجيب مكافأة قد إيه على المدى البعيد". وبيحدّثه بعد كل خطوة:

$$ Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,] $$

وبيوازن بين **الاستكشاف** (يجرب حركة عشوائية بنسبة ε) و**الاستغلال** (ياخد أحسن حركة يعرفها).

</div>

In [ ]:
def env_step(pos, action):
    dr, dc = ACTIONS[action]
    nxt = (pos[0] + dr, pos[1] + dc)
    if not is_free(nxt):
        return pos, -5, False
    if nxt == goal:
        return nxt, 50, True
    return nxt, -1, False


def q_learning(episodes=400, alpha=0.5, gamma=0.95, eps_start=1.0, eps_end=0.05, seed=0):
    rng = np.random.default_rng(seed)
    actions = list(ACTIONS)
    Q = np.zeros((H, W, len(actions)))
    steps_per_episode = []
    for ep in range(episodes):
        eps = max(eps_end, eps_start * (1 - ep / (episodes * 0.7)))
        pos, done, steps = start, False, 0
        while not done and steps < 300:
            a = rng.integers(len(actions)) if rng.random() < eps else int(np.argmax(Q[pos]))
            nxt, reward, done = env_step(pos, actions[a])
            target = reward + (0 if done else gamma * Q[nxt].max())
            Q[pos][a] += alpha * (target - Q[pos][a])
            pos, steps = nxt, steps + 1
        steps_per_episode.append(steps)
    return Q, steps_per_episode

Q, steps_per_episode = q_learning()

plt.figure(figsize=(7, 3.5))
plt.plot(steps_per_episode, alpha=0.4)
plt.plot(np.convolve(steps_per_episode, np.ones(20) / 20, mode="valid"), linewidth=2)
plt.axhline(len(path) - 1, color="red", linestyle="--", label="BFS optimum")
plt.xlabel("Episode (try)")
plt.ylabel("Steps to reach the goal")
plt.yscale("log")
plt.legend()
plt.title("The robot learns: fewer steps every try")
plt.show()

In [ ]:
def greedy_path(Q, max_steps=100):
    actions, pos, route = list(ACTIONS), start, [start]
    for _ in range(max_steps):
        pos, _, done = env_step(pos, actions[int(np.argmax(Q[pos]))])
        route.append(pos)
        if done:
            return route
    return route

learned = greedy_path(Q)
print("Learned path length:", len(learned) - 1, "steps | BFS:", len(path) - 1)
draw(learned, values=Q.max(axis=2), title="Learned path + how 'good' each cell looks to the robot")

<div dir="rtl">

**الألوان:** كل ما الخانة تفتح، الروبوت شايف إنها أقرب للهدف. الروبوت **بنى خريطة في دماغه** من التجربة بس!

## BFS ولا RL؟
| | BFS | Q-Learning |
|---|---|---|
| محتاج الخريطة؟ | أيوه | لأ |
| سريع؟ | جداً | محتاج مئات المحاولات |
| ينفع في عالم مش معروف/بيتغير؟ | لأ | أيوه |
| أمثلة | Google Maps | روبوتات بتتعلم تمشي، AlphaGo |

</div>

<div dir="rtl">

## 🏋️ تمرين ١
حط حيطة جديدة في الخريطة تقفل الطريق اللي لقاه BFS، وشغّل BFS تاني. لقى طريق تاني؟

<details><summary>الحل</summary>

```python
r, c = path[len(path) // 2]
grid[r, c] = "#"
new_path = bfs(start, goal)
print("New path:", None if new_path is None else len(new_path) - 1, "steps")
if new_path:
    draw(new_path, title="Re-planned path")
grid[r, c] = "."   # نرجّع الخريطة زي ما كانت
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢
غيّر عقاب الخبط في الحيطة لـ `0` وعقاب الخطوة لـ `0`. هل الروبوت لسه بيتعلم أقصر طريق؟ ليه؟ وهل بيتعلم بنفس السرعة؟

<details><summary>الحل</summary>

```python
def env_step(pos, action):
    dr, dc = ACTIONS[action]
    nxt = (pos[0] + dr, pos[1] + dc)
    if not is_free(nxt):
        return pos, 0, False
    if nxt == goal:
        return nxt, 50, True
    return nxt, 0, False

Q2, steps2 = q_learning()
route = greedy_path(Q2)
print("Reached goal:", route[-1] == goal, "| path length:", len(route) - 1)
print("Avg steps in first 100 tries - with penalties:", int(np.mean(steps_per_episode[:100])),
      "| without:", int(np.mean(steps2[:100])))
# النتيجة: لسه بيوصل لأقصر طريق في الآخر! السبب هو gamma (0.95): المكافأة اللي بتيجي متأخر قيمتها أقل،
# فالطريق الأقصر لسه "أكسب". بس قارن متوسط الخطوات في أول محاولات علشان تشوف تأثير العقاب على سرعة التعلم.
# جرّب كمان gamma = 1.0 من غير عقاب: ساعتها الطويل والقصير قيمتهم واحدة.
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ✅ لخّص
- **Planning (BFS):** عندك خريطة ← احسب الطريق.
- **Reinforcement Learning:** مفيش خريطة ← جرّب، خد مكافأة، اتعلم.
- **Reward design** (تصميم المكافأة) بيأثر جامد على اللي الروبوت هيتعلمه.
- **IoT + AI:** الروبوت الحقيقي بيجمع البيانات من حساسات (مسافة، كاميرا) بدل الخريطة.

</div>